# LightRAG Document Ingestion

This notebook implements a lightweight RAG system for processing PDF and text documents using local Ollama embeddings.

## Setup

First, install required packages:

In [ ]:
!pip install numpy requests PyPDF2 tqdm

## Import Dependencies

In [27]:
import os
from pathlib import Path
from typing import List, Dict, Union
import PyPDF2
import numpy as np
import json
import pickle
from tqdm import tqdm
import requests

## Implement Ollama Embedding Client

In [28]:
class OllamaEmbedding:
    def __init__(self, model_name: str = "nomic-embed-text", base_url: str = "http://localhost:11434"):
        """Initialize Ollama embedding client."""
        self.model_name = model_name
        self.base_url = base_url
    
    def encode(self, texts: Union[str, List[str]]) -> np.ndarray:
        """Generate embeddings for text using Ollama API."""
        if isinstance(texts, str):
            texts = [texts]
        
        embeddings = []
        for text in texts:
            response = requests.post(
                f"{self.base_url}/api/embeddings",
                json={
                    "model": self.model_name,
                    "prompt": text
                }
            )
            embeddings.append(response.json()['embedding'])
        
        return np.array(embeddings)

## Document Processing Functions

In [29]:
def read_pdf(file_path: str) -> str:
    """Read content from a PDF file."""
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        text = ''
        for page in pdf_reader.pages:
            text += page.extract_text() + '\n'
    return text

def read_text(file_path: str) -> str:
    """Read content from a text file."""
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def chunk_text(text: str, chunk_size: int = 512, overlap: int = 50) -> List[str]:
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    return chunks

## LightRAG Implementation

In [30]:
class LightRAG:
    def __init__(self, embedding_provider: str = "ollama", 
                 model_name: str = "nomic-embed-text"):
        """Initialize LightRAG with chosen embedding provider."""
        if embedding_provider == "ollama":
            self.model = OllamaEmbedding(model_name)
        else:
            raise ValueError("Unsupported embedding provider")
            
        self.documents = []
        self.embeddings = None
        self.metadata = []
        
    def add_document(self, content: str, metadata: Dict):
        """Add a document and its metadata to the RAG system."""
        chunks = chunk_text(content)
        
        # Create embeddings for chunks
        chunk_embeddings = self.model.encode(chunks)
        
        # Store chunks and their metadata
        if self.embeddings is None:
            self.embeddings = chunk_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, chunk_embeddings])
        
        # Store document chunks and metadata
        for chunk in chunks:
            self.documents.append(chunk)
            self.metadata.append(metadata)
    
    def save(self, directory: str):
        """Save the RAG system to disk."""
        os.makedirs(directory, exist_ok=True)
        
        # Save embeddings
        np.save(os.path.join(directory, 'embeddings.npy'), self.embeddings)
        
        # Save documents and metadata
        with open(os.path.join(directory, 'documents.pkl'), 'wb') as f:
            pickle.dump({'documents': self.documents, 'metadata': self.metadata}, f)

    def list_documents(self) -> List[str]:
        """List all documents in the RAG system."""
        return self.documents
    
    def delete_document(self, index: int):
        """Delete a document from the RAG system."""
        self.documents.pop(index)
        self.metadata.pop(index)
        self.embeddings = np.delete(self.embeddings, index, axis=0)

## Process Documents

Now let's process all documents in the specified directory:

In [ ]:
import os
from pathlib import Path
import pickle
import numpy as np

def get_existing_documents(index_directory: str) -> dict:
    """Load existing document metadata from the index"""
    try:
        with open(f"{index_directory}/documents.pkl", 'rb') as f:
            data = pickle.load(f)
            return {doc['path']: idx for idx, doc in enumerate(data['metadata'])}
    except FileNotFoundError:
        return {}

def update_rag_index(docs_dir: str = "documents/", index_dir: str = "lightrag_index"):
    # Get current files in documents directory
    current_files = set()
    for file_path in Path(docs_dir).rglob('*'):
        if file_path.suffix.lower() in ['.pdf', '.txt']:
            current_files.add(str(file_path))

    # Check if RAG index exists
    if os.path.exists(index_dir):
        # Load existing document paths
        existing_docs = get_existing_documents(index_dir)
        existing_paths = set(existing_docs.keys())

        # Find new and removed files
        new_files = current_files - existing_paths
        removed_files = existing_paths - current_files

        if not new_files and not removed_files:
            print("Index is up to date - no changes needed")
            return

        # Load existing RAG
        rag = LightRAG()
        rag.embeddings = np.load(f"{index_dir}/embeddings.npy")
        with open(f"{index_dir}/documents.pkl", 'rb') as f:
            data = pickle.load(f)
            rag.documents = data['documents']
            rag.metadata = data['metadata']

        # Remove missing documents
        for file_path in removed_files:
            idx = existing_docs[file_path]
            print(f"Removing {file_path} from index")
            rag.delete_document(idx)

        # Add new documents
        for file_path in new_files:
            print(f"Adding {file_path} to index")
            if file_path.endswith('.pdf'):
                content = read_pdf(file_path)
            else:
                content = read_text(file_path)

            metadata = {
                'filename': Path(file_path).name,
                'path': str(file_path),
                'type': Path(file_path).suffix.lower()
            }
            rag.add_document(content, metadata)

        # Save updated index
        rag.save(index_dir)
        print("Index updated successfully")

    else:
        # Create new RAG index
        print("Creating new RAG index")
        rag = LightRAG()
        
        for file_path in current_files:
            print(f"Processing {file_path}")
            if str(file_path).endswith('.pdf'):
                content = read_pdf(str(file_path))
            else:
                content = read_text(str(file_path))

            metadata = {
                'filename': Path(file_path).name,
                'path': str(file_path),
                'type': Path(file_path).suffix.lower()
            }
            rag.add_document(content, metadata)

        rag.save(index_dir)
        print("New index created successfully")

update_rag_index()